# ODI Complaints Cleaning

Use this notebook to test candidate cleaning rules before we lock them into the `src/` pipeline. The goal is to make cleaning choices that match the schema docs, the issues we found in EDA, and the kinds of features we expect to build later.


## Cleaning Principles

- Keep raw meaning whenever possible
- Normalize fields that will be grouped or joined
- Convert true measurements to numeric types
- Convert encoded dates to datetimes
- Flag suspicious records before dropping anything
- Avoid cleaning choices that would erase useful safety signal


In [ ]:
# Imports
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 120)

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("deep")


## Load The Combined Processed Dataset


In [ ]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "processed").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

COMBINED_PATH = PROJECT_ROOT / "data" / "processed" / "odi_complaints_combined.parquet"

df = pd.read_parquet(COMBINED_PATH)
df = df.drop(columns=["source_zip", "source_file"], errors="ignore")

print("Loaded:", COMBINED_PATH.name)
print("Shape:", df.shape)


## CMPL.txt Notes That Matter For Cleaning

This notebook treats `docs/CMPL.txt` as the source contract for field meaning. That matters because some cleaning decisions change once the documented definitions are taken seriously.


In [ ]:
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data.schema_checks import get_schema_spec

spec = get_schema_spec("complaints")
schema_df = pd.DataFrame(spec["fields"])
focus_cols = [
    "yeartxt",
    "faildate",
    "datea",
    "ldate",
    "miles",
    "injured",
    "deaths",
    "veh_speed",
    "vin",
    "purch_dt",
    "manuf_dt",
    "state",
    "dealer_state",
    "prod_type"
]

schema_view = schema_df.loc[schema_df["name"].isin(focus_cols), ["name", "type", "size", "description"]].copy()
schema_view = schema_view.sort_values("name")

change_notes = pd.DataFrame(
    [
        {
            "note": "May-Jun 2021 file update",
            "implication": "Previously blank Y/N fields may now appear as N"
        },
        {
            "note": "May-Jun 2021 file update",
            "implication": "Previously blank numeric fields may now appear as 0"
        },
        {
            "note": "May-Jun 2021 file update",
            "implication": "Manufacturer, make, model, and component values may reflect newer cleanup over time"
        }
    ]
)

display(schema_view)
display(change_notes)


## Baseline Snapshot


In [ ]:
baseline = pd.DataFrame(
    {
        "metric": [
            "rows",
            "columns",
            "cmplid unique",
            "odino unique",
            "full-row duplicates"
        ],
        "value": [
            len(df),
            df.shape[1],
            df["cmplid"].nunique(dropna=True),
            df["odino"].nunique(dropna=True),
            int(df.duplicated().sum())
        ]
    }
)

dtype_ct = (
    df.dtypes.astype("string")
    .value_counts()
    .rename_axis("dtype")
    .reset_index(name="column_count")
)

null_top = (
    df.isna()
    .mean()
    .mul(100)
    .round(2)
    .sort_values(ascending=False)
    .head(15)
    .rename("null_pct")
    .reset_index()
    .rename(columns={"index": "column"})
)

display(baseline)
display(dtype_ct)
display(null_top)


## Candidate Rule Map


In [ ]:
rule_map = pd.DataFrame(
    [
        {"column": "cmplid", "candidate_action": "keep as string", "why": "row-level complaint key"},
        {"column": "odino", "candidate_action": "keep as string", "why": "case reference and not unique"},
        {"column": "maketxt / modeltxt / city / dealer_city / compdesc", "candidate_action": "strip and uppercase", "why": "used for grouping and joins"},
        {"column": "state / dealer_state", "candidate_action": "strip, uppercase, validate", "why": "categorical location codes"},
        {"column": "cdescr", "candidate_action": "strip only", "why": "preserve narrative signal for NLP"},
        {"column": "vin", "candidate_action": "strip and uppercase", "why": "identifier-like text defined as CHAR(11) in CMPL.txt"},
        {"column": "yeartxt", "candidate_action": "convert to Int64 and map 9999 to missing", "why": "true model-year field with documented unknown code"},
        {"column": "injured / deaths / occurences / num_cyls / veh_speed / miles", "candidate_action": "convert to Int64", "why": "measurements or counts used in features"},
        {"column": "purch_dt / manuf_dt", "candidate_action": "convert to datetime", "why": "encoded dates stored as YYYYMMDD"},
        {"column": "dealer_zip / dealer_tel", "candidate_action": "keep as string", "why": "identifiers that can lose meaning if coerced to numeric"},
        {"column": "bad chronology / bad state / bad vin length", "candidate_action": "flag first", "why": "review before any drop decision"}
    ]
)

display(rule_map)


## Setup Candidate Cleaning Frame


In [ ]:
clean = df.copy()

YN_COLS = [
    "crash",
    "fire",
    "police_rpt_yn",
    "orig_owner_yn",
    "anti_brakes_yn",
    "cruise_cont_yn",
    "orig_equip_yn",
    "repaired_yn",
    "medical_attn",
    "vehicles_towed_yn"
]

UPPER_COLS = [
    "mfr_name",
    "maketxt",
    "modeltxt",
    "compdesc",
    "city",
    "state",
    "vin",
    "cmpl_type",
    "dealer_name",
    "dealer_city",
    "dealer_state",
    "prod_type"
] + YN_COLS

STRIP_ONLY_COLS = [
    "cdescr",
    "fuel_sys",
    "fuel_type",
    "trans_type",
    "drive_train",
    "dot",
    "tire_size",
    "loc_of_tire",
    "tire_fail_type",
    "dealer_zip",
    "dealer_tel"
]

INT_COLS = [
    "yeartxt",
    "injured",
    "deaths",
    "miles",
    "occurences",
    "num_cyls",
    "veh_speed"
]

DATE_STR_COLS = [
    "purch_dt",
    "manuf_dt"
]

POSTAL_CODES = {
    "AL", "AK", "AZ", "AR", "CA", "CO", "CT", "DE", "FL", "GA", "HI", "ID", "IL", "IN", "IA", "KS", "KY", "LA", "ME", "MD",
    "MA", "MI", "MN", "MS", "MO", "MT", "NE", "NV", "NH", "NJ", "NM", "NY", "NC", "ND", "OH", "OK", "OR", "PA", "RI", "SC",
    "SD", "TN", "TX", "UT", "VT", "VA", "WA", "WV", "WI", "WY", "DC", "PR", "VI", "GU", "AS", "MP", "FM", "MH", "PW", "AE",
    "AA", "AP"
}

# -----------------------------------------------------------------------------
# Small helpers
# -----------------------------------------------------------------------------
def strip_text(series):
    text = series.astype("string").str.strip()
    return text.replace({"": pd.NA})


def upper_text(series):
    return strip_text(series).str.upper()


def to_int(series):
    nums = pd.to_numeric(strip_text(series), errors="coerce")
    return nums.astype("Int64")


def to_dt(series):
    return pd.to_datetime(strip_text(series), format="%Y%m%d", errors="coerce")


def changed_rows(before, after):
    return int(before.astype("string").fillna("<NA>").ne(after.astype("string").fillna("<NA>")).sum())


step_log = []


## Step 1 Normalize Text And Code Fields


In [ ]:
step1_rows = []

for col in UPPER_COLS:
    if col not in clean.columns:
        continue
    before = clean[col].copy()
    clean[col] = upper_text(clean[col])
    step1_rows.append(
        {
            "column": col,
            "action": "strip + upper",
            "changed_rows": changed_rows(before, clean[col]),
            "null_after": int(clean[col].isna().sum())
        }
    )

for col in STRIP_ONLY_COLS:
    if col not in clean.columns:
        continue
    before = clean[col].copy()
    clean[col] = strip_text(clean[col])
    step1_rows.append(
        {
            "column": col,
            "action": "strip only",
            "changed_rows": changed_rows(before, clean[col]),
            "null_after": int(clean[col].isna().sum())
        }
    )

step1 = pd.DataFrame(step1_rows).sort_values(["changed_rows", "column"], ascending=[False, True])
step_log.append({"step": "normalize text and code fields", "rows_changed": int(step1["changed_rows"].sum())})

display(step1[step1["changed_rows"] > 0])


## Step 2 Convert True Dates And Measurements


In [ ]:
step2_rows = []

for col in DATE_STR_COLS:
    if col not in clean.columns:
        continue
    before_dtype = str(clean[col].dtype)
    non_null_before = int(clean[col].notna().sum())
    clean[col] = to_dt(clean[col])
    step2_rows.append(
        {
            "column": col,
            "action": "to datetime",
            "dtype_before": before_dtype,
            "dtype_after": str(clean[col].dtype),
            "non_null_before": non_null_before,
            "non_null_after": int(clean[col].notna().sum())
        }
    )

for col in INT_COLS:
    if col not in clean.columns:
        continue
    before_dtype = str(clean[col].dtype)
    non_null_before = int(clean[col].notna().sum())
    clean[col] = to_int(clean[col])
    step2_rows.append(
        {
            "column": col,
            "action": "to Int64",
            "dtype_before": before_dtype,
            "dtype_after": str(clean[col].dtype),
            "non_null_before": non_null_before,
            "non_null_after": int(clean[col].notna().sum())
        }
    )

step2 = pd.DataFrame(step2_rows)
step_log.append({"step": "convert dates and measurements", "rows_changed": int(step2["non_null_after"].sum())})

display(step2)


## Step 3 Flag Suspicious Records And Map Candidate Missing Values


In [ ]:
current_year = pd.Timestamp.today().year

clean["flag_year_unknown"] = clean["yeartxt"].eq(9999)
clean["flag_year_out_of_range"] = clean["yeartxt"].notna() & ~clean["yeartxt"].between(1900, current_year + 1)
clean["flag_speed_bad"] = clean["veh_speed"].eq(999) | clean["veh_speed"].gt(200)
clean["flag_miles_high"] = clean["miles"].gt(500000)
clean["flag_injured_bad"] = clean["injured"].eq(99)
clean["flag_deaths_bad"] = clean["deaths"].eq(99)
clean["flag_state_bad"] = clean["state"].notna() & ~clean["state"].isin(POSTAL_CODES)
clean["flag_dealer_state_bad"] = clean["dealer_state"].notna() & ~clean["dealer_state"].isin(POSTAL_CODES)
clean["flag_vin_len_bad"] = clean["vin"].notna() & clean["vin"].str.len().ne(11)

# CMPL.txt says DATEA is date added to file and LDATE is date received by NHTSA
clean["flag_fail_after_added"] = clean["faildate"].notna() & clean["datea"].notna() & (clean["faildate"] > clean["datea"])
clean["flag_fail_after_received"] = clean["faildate"].notna() & clean["ldate"].notna() & (clean["faildate"] > clean["ldate"])
clean["flag_added_before_received"] = clean["datea"].notna() & clean["ldate"].notna() & (clean["datea"] < clean["ldate"])
clean["flag_date_order_bad"] = clean[["flag_fail_after_added", "flag_fail_after_received", "flag_added_before_received"]].any(axis=1)

clean.loc[clean["flag_year_unknown"] | clean["flag_year_out_of_range"], "yeartxt"] = pd.NA
clean.loc[clean["flag_speed_bad"], "veh_speed"] = pd.NA
clean.loc[clean["flag_miles_high"], "miles"] = pd.NA
clean.loc[clean["flag_injured_bad"], "injured"] = pd.NA
clean.loc[clean["flag_deaths_bad"], "deaths"] = pd.NA
clean.loc[clean["flag_state_bad"], "state"] = pd.NA
clean.loc[clean["flag_dealer_state_bad"], "dealer_state"] = pd.NA

flag_summary = pd.DataFrame(
    {
        "flag": [
            "flag_year_unknown",
            "flag_year_out_of_range",
            "flag_speed_bad",
            "flag_miles_high",
            "flag_injured_bad",
            "flag_deaths_bad",
            "flag_state_bad",
            "flag_dealer_state_bad",
            "flag_vin_len_bad",
            "flag_fail_after_added",
            "flag_fail_after_received",
            "flag_added_before_received",
            "flag_date_order_bad"
        ]
    }
)
flag_summary["row_count"] = flag_summary["flag"].map(lambda col: int(clean[col].sum()))

step_log.append({"step": "flag sentinels and suspicious records", "rows_changed": int(flag_summary["row_count"].sum())})

display(flag_summary)


## Compare Before And After


In [ ]:
compare_cols = [
    "yeartxt",
    "miles",
    "veh_speed",
    "injured",
    "deaths",
    "purch_dt",
    "manuf_dt",
    "state",
    "dealer_state",
    "maketxt",
    "modeltxt",
    "city",
    "dealer_city"
]

na_compare = pd.DataFrame(
    {
        "before_null_pct": (df[compare_cols].isna().mean() * 100).round(2),
        "after_null_pct": (clean[compare_cols].isna().mean() * 100).round(2)
    }
).reset_index().rename(columns={"index": "column"})

display(na_compare.sort_values("after_null_pct", ascending=False))

plot_df = na_compare.melt(id_vars="column", var_name="stage", value_name="null_pct")

plt.figure(figsize=(12, 8))
sns.barplot(data=plot_df, x="null_pct", y="column", hue="stage")
plt.title("Null percentage before and after candidate cleaning")
plt.xlabel("null percentage")
plt.ylabel("")
plt.tight_layout()
plt.show()


## Output Checks


In [ ]:
step_log_df = pd.DataFrame(step_log)

dtype_after = (
    clean[["yeartxt", "injured", "deaths", "miles", "occurences", "num_cyls", "veh_speed", "purch_dt", "manuf_dt"]]
    .dtypes.astype("string")
    .rename("dtype")
    .reset_index()
    .rename(columns={"index": "column"})
)

checks = pd.DataFrame(
    {
        "metric": [
            "rows after cleaning",
            "columns after cleaning",
            "cmplid unique after cleaning",
            "odino unique after cleaning",
            "full-row duplicates after cleaning"
        ],
        "value": [
            len(clean),
            clean.shape[1],
            clean["cmplid"].nunique(dropna=True),
            clean["odino"].nunique(dropna=True),
            int(clean.duplicated().sum())
        ]
    }
)

display(step_log_df)
display(dtype_after)
display(checks)


## 2021 Change-Log Watchout


In [ ]:
zero_watch = (
    clean.assign(received_year=clean["ldate"].dt.year)
    .groupby("received_year")[["injured", "deaths", "miles", "veh_speed"]]
    .agg(lambda s: round(float((s == 0).mean() * 100), 2))
    .reset_index()
)

print("Percent of rows equal to zero by received year")
display(zero_watch)


## Rows Worth Inspecting By Hand


In [ ]:
inspect_cols = [
    "cmplid",
    "odino",
    "maketxt",
    "modeltxt",
    "yeartxt",
    "state",
    "dealer_state",
    "vin",
    "faildate",
    "datea",
    "ldate",
    "miles",
    "veh_speed",
    "injured",
    "deaths",
    "cdescr"
]

print("Bad chronology examples")
display(clean.loc[clean["flag_date_order_bad"], inspect_cols].head(10))

print("Bad state examples")
display(clean.loc[clean["flag_state_bad"] | clean["flag_dealer_state_bad"], inspect_cols].head(10))

print("High miles or bad speed examples")
display(clean.loc[clean["flag_miles_high"] | clean["flag_speed_bad"], inspect_cols].head(10))

print("VIN length examples")
display(clean.loc[clean["flag_vin_len_bad"], inspect_cols].head(10))


## Scratchpad


In [ ]:
# Example checks
# clean.columns.tolist()
# clean[clean["flag_state_bad"]][["state", "city"]].head(20)
# clean.groupby("prod_type").size().sort_values(ascending=False)
# clean[clean["fire"] == "Y"][["maketxt", "modeltxt", "compdesc"]].head(20)
# clean.to_parquet(PROJECT_ROOT / "data" / "outputs" / "clean_candidate.parquet", index=False)
